# 第十九课:RoPE——旋转的位置编码

讲义:[19-rope.md](19-rope.md)。

In [ ]:
import numpy as np, math

# 二维旋转(单频率 θ=1):核心恒等式验证
rot = lambda v, th: np.array([v[0]*math.cos(th) - v[1]*math.sin(th),
                              v[0]*math.sin(th) + v[1]*math.cos(th)])
q, k = np.array([1.0, 2.0]), np.array([3.0, 4.0])

for m, n in [(0, 0), (0, 1), (3, 1), (5, 2)]:
    lhs = rot(q, m) @ rot(k, n)
    rhs = q @ rot(k, n - m)
    print(f'm={m} n={n}: {lhs:.6f} vs {rhs:.6f}  一致: {np.isclose(lhs, rhs)}')

In [ ]:
def rope_apply(x, pos):
    n, d = x.shape
    half = d // 2
    theta = 1.0 / (10000 ** (np.arange(half) * 2 / d))
    ang = pos[:, None] * theta[None, :]
    c, s = np.cos(ang), np.sin(ang)
    x1, x2 = x[:, :half], x[:, half:]
    return np.concatenate([x1*c - x2*s, x1*s + x2*c], axis=-1)

x = np.random.default_rng(0).normal(size=(3, 4))
out = rope_apply(x, np.arange(3))
print('形状:', x.shape, '->', out.shape, ' 有限:', np.isfinite(out).all())

# 范数不变性:旋转不改变向量长度
print('旋转前范数:', np.round(np.linalg.norm(x, axis=1), 4))
print('旋转后范数:', np.round(np.linalg.norm(out, axis=1), 4))

In [ ]:
# 频率谱:高频管近距离,低频管远距离(作业 E1)
d = 64
theta = 1.0 / (10000 ** (np.arange(d//2) * 2 / d))
wavelength = 2 * np.pi / theta
print('最高频波长: %.2f rad' % wavelength[0])
print('最低频波长: %.3e rad' % wavelength[-1])
print('跨越数量级:', round(math.log10(wavelength[-1]/wavelength[0]), 1))

In [ ]:
# TODO(C2): 5 组 (m,n) 验证恒等式,误差 < 1e-10
# TODO(C3): 第六课玩具 + RoPE(投影→旋转→打分),对比权重矩阵
# TODO(D2): theta_scaled(theta, factor) —— linear scaling
